# 02 · Inspección
Primera revisión de la temporada 2025-26: forma, tipos, nulos y granularidad.

**Pregunta de este notebook:** ¿puedo confiar en esta tabla y entiendo qué representa cada fila?
Es el primer paso para responder: ¿qué diferencia a los equipos que ganan de los que pierden?

In [2]:
import pandas as pd

df = pd.read_parquet("../data/raw/team_game_log_2025_26.parquet")
df.sample(5, random_state=1)

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
1268,22025,1610612744,GSW,Golden State Warriors,0022500612,2026-01-19,GSW vs. MIA,W,240,49,...,38,46,38,11,3,14,17,135,23,1
2270,22025,1610612741,CHI,Chicago Bulls,0022501109,2026-04-01,CHI vs. IND,L,240,47,...,32,46,22,6,4,20,14,126,-19,1
1617,22025,1610612752,NYK,New York Knicks,0022500780,2026-02-11,NYK @ PHI,W,240,48,...,39,51,41,13,6,15,16,138,49,1
499,22025,1610612766,CHA,Charlotte Hornets,0022500276,2025-11-23,CHA @ ATL,L,240,39,...,34,43,26,4,2,16,15,110,-3,1
535,22025,1610612756,PHX,Phoenix Suns,0022500068,2025-11-26,PHX @ SAC,W,240,42,...,30,47,25,14,5,17,14,112,12,1


A primera vista, cada fila corresponde a un equipo en un partido. La tabla tiene tres identificadores con propósitos distintos: SEASON_ID (la temporada), TEAM_ID (el equipo) y GAME_ID (el partido). Es una tabla plana o desnormalizada: el nombre y la sigla del equipo se repiten en cada uno de sus partidos, lo que facilita el análisis porque no hay que unir tablas.

Además de la fecha (GAME_DATE), cada fila trae las estadísticas del equipo en ese partido (tiros, rebotes, asistencias y puntos) y PLUS_MINUS, que para un equipo es la diferencia final de puntos. MATCHUP indica si jugó de local (vs.) o de visita (@), y MIN cuántos minutos jugó. VIDEO_AVAILABLE no aporta a las preguntas de este análisis.

## 2. ¿De qué tipo es cada columna?
revisaremos que tipo de datos tiene cada columna para verificar si necesitaremos alguna transformacion, fecha puede venir en string y no en date, o los porcentajes de tiro pueden venir en otro fomato que no sea en float64 con esto lo corroboramos 

In [3]:
df.dtypes

SEASON_ID                str
TEAM_ID                int64
TEAM_ABBREVIATION        str
TEAM_NAME                str
GAME_ID                  str
GAME_DATE                str
MATCHUP                  str
WL                       str
MIN                    int64
FGM                    int64
FGA                    int64
FG_PCT               float64
FG3M                   int64
FG3A                   int64
FG3_PCT              float64
FTM                    int64
FTA                    int64
FT_PCT               float64
OREB                   int64
DREB                   int64
REB                    int64
AST                    int64
STL                    int64
BLK                    int64
TOV                    int64
PF                     int64
PTS                    int64
PLUS_MINUS             int64
VIDEO_AVAILABLE        int64
dtype: object

Los tipos calzan con el contenido de cada columna:

Texto (str): los identificadores SEASON_ID y GAME_ID, la sigla y el nombre del equipo, MATCHUP, WL y GAME_DATE.
Enteros (int64): las estadísticas que se cuentan (tiros, rebotes, asistencias, puntos), MIN y PLUS_MINUS.
Decimales (float64): los tres porcentajes de tiro, escritos como fracción entre 0 y 1.

SEASON_ID y GAME_ID están bien como texto: son códigos, no cantidades. TEAM_ID también es un identificador aunque venga como número; no causa problemas, pero no se usa en cálculos.

El único tipo que hay que corregir es el de GAME_DATE: como texto, no permite filtrar por mes ni ordenar por calendario.

## 3. ¿Faltan datos?

In [6]:
df.isna().sum()

SEASON_ID            0
TEAM_ID              0
TEAM_ABBREVIATION    0
TEAM_NAME            0
GAME_ID              0
GAME_DATE            0
MATCHUP              0
WL                   0
MIN                  0
FGM                  0
FGA                  0
FG_PCT               0
FG3M                 0
FG3A                 0
FG3_PCT              0
FTM                  0
FTA                  0
FT_PCT               0
OREB                 0
DREB                 0
REB                  0
AST                  0
STL                  0
BLK                  0
TOV                  0
PF                   0
PTS                  0
PLUS_MINUS           0
VIDEO_AVAILABLE      0
dtype: int64

Ninguna columna tiene valores vacíos: la API entrega la tabla completa a nivel de celda. Esto no garantiza que estén todos los partidos (una fila que falta no deja celdas vacías), y eso se revisa en la sección 4. Tampoco descarta nulos disfrazados, como un porcentaje en 0 cuando no hubo intentos, que se revisan en la limpieza.

## 4. ¿Qué representa una fila y qué la identifica?

In [27]:
print("filas", len(df)) # funcion len() devuelve la cantidad de filas
print("partidos distintos", df["GAME_ID"].nunique()) #contar la cantidad de partidos distintos
print("filas por partido", df.groupby("GAME_ID").size().unique()) #se agrupa por game id, size cuenta las filas de cada grupo, unique() cantidades distintas aparecieron
print("partidos por equipo", df["TEAM_ABBREVIATION"].value_counts().unique()) #cuenta la cantidad de las abreviaciones que aparecen en el df y luego cuenta los partidos unicos ya que cada fila es un partido 
print("filas repetidas (GAME_ID + TEAM_ID):", df.duplicated(subset=["GAME_ID","TEAM_ID"]).sum())

filas 2460
partidos distintos 1230
filas por partido [2]
partidos por equipo [82]
filas repetidas (GAME_ID + TEAM_ID): 0


Cada fila representa a un equipo en un partido. La tabla tiene 2.460 filas: 30 equipos × 82 partidos, así que no falta ningún partido (en temporadas con lockout no serían 82). Hay 1.230 partidos distintos, la mitad de las filas, porque cada partido aparece dos veces, una por cada equipo. Si Miami jugó contra Indiana, hay una fila de Miami y otra de Indiana.

Las comprobaciones lo confirman: todos los partidos tienen exactamente 2 filas y todos los equipos jugaron 82. La llave de la tabla es la combinación de GAME_ID y TEAM_ID: ninguna fila repite esa pareja.

Consecuencia para el análisis: contar filas no es contar partidos. Cualquier cálculo por partido tiene que tomar en cuenta que cada uno aparece dos veces.

## 5. ¿Qué cambia entre las dos filas de un mismo partido?

In [36]:
un_partido = df["GAME_ID"].iloc[0]
df[df["GAME_ID"] == un_partido].T


,0,2
SEASON_ID,22025,22025
TEAM_ID,1610612744,1610612747
TEAM_ABBREVIATION,GSW,LAL
TEAM_NAME,Golden State Warriors,Los Angeles Lakers
GAME_ID,0022500002,0022500002
GAME_DATE,2025-10-21,2025-10-21
MATCHUP,GSW @ LAL,LAL vs. GSW
WL,W,L
MIN,240,240
FGM,38,42


En las dos filas de un partido se repiten las columnas que describen el partido: SEASON_ID, GAME_ID, GAME_DATE y MIN. Cambian las que describen a cada equipo: sus identificadores, sus estadísticas, MATCHUP, WL y PLUS_MINUS.

Ejemplo: el partido inaugural del 21 de octubre de 2025, en que GSW ganó 119 a 109 en cancha de LAL.

Local o visita: MATCHUP dice GSW @ LAL en la fila de GSW y LAL vs. GSW en la de LAL. El @ marca al visitante y vs. al local.
Resultado: PLUS_MINUS es la diferencia final de puntos, con el mismo valor y signo contrario en cada fila (+10 y −10). Su signo coincide siempre con WL, lo que sirve para validar los datos.
Puntos del rival: se calculan como PTS − PLUS_MINUS. Para GSW, 119 − 10 = 109; para LAL, 109 − (−10) = 119.
Minutos: MIN = 240 corresponde a un partido sin prórroga (48 minutos × 5 jugadores). Cada prórroga suma 25.

## Conclusión general

**Sí, puedo confiar en esta tabla para el análisis**, con tres cuidados que resuelvo en la limpieza.

- **Está completa.** Tiene 2.460 filas (30 equipos × 82 partidos): no falta ningún partido y ninguna celda está vacía.
- **Cada fila es un equipo en un partido.** La llave es `GAME_ID` + `TEAM_ID`, y cada partido aparece dos veces, una por equipo. Por eso, contar filas no es contar partidos.
- **Los tipos son correctos**, salvo `GAME_DATE`, que viene como texto.

**Los tres cuidados:**
1. Convertir la fecha.
2. Revisar si hay nulos disfrazados, como porcentajes en 0 sin intentos.
3. Crear las columnas que la tabla no trae directamente (local o visita, resultado, puntos en contra y prórrogas) y validar que estén bien calculadas.

## Próximo paso

En el notebook `03_limpieza` aplico las ocho decisiones de esta inspección y guardo una tabla limpia. La copia raw no se modifica.

## Decisiones para la limpieza

Lo que encontré en la inspección y lo que voy a hacer en el notebook 03.

| # | Decisión | Sale de | Por qué |
|---|---|---|---|
| 1 | Convertir `GAME_DATE` a fecha | Sección 2 | Como texto no se puede filtrar por mes ni ordenar por calendario |
| 2 | Crear `gano`: 1 si `WL` es `W` y 0 si es `L` | Sección 2 | El promedio de una columna de unos y ceros es el porcentaje de victorias |
| 3 | No usar `VIDEO_AVAILABLE` | Sección 1 | No aporta a ninguna de las preguntas del análisis |
| 4 | Revisar si hay porcentajes en 0 con 0 intentos | Sección 3 | Serían nulos disfrazados: "no hubo intentos" no es lo mismo que "falló todos" |
| 5 | Crear `es_local`: 1 si `MATCHUP` contiene `vs.` y 0 si contiene `@` | Sección 5 | Se necesita para medir la ventaja de local (pregunta 1) |
| 6 | Crear `puntos_en_contra = PTS − PLUS_MINUS` y validar que coincida con los `PTS` del rival | Sección 5 | Se necesita para medir qué tan parejos son los partidos (pregunta 3) |
| 7 | Crear `prorrogas = (MIN − 240) / 25` | Sección 5 | Cada prórroga suma 25 minutos de equipo; sirve para identificar partidos que se fueron a tiempo extra |
| 8 | Validar que cada partido tenga un `W` y un `L`, y que `W` coincida con `PLUS_MINUS` positivo | Secciones 4 y 5 | Si dos columnas que deben decir lo mismo no coinciden, hay un error en los datos |